In [ ]:
# 新数据集标准化与baseline模型预测
# 适配 Jupyter Notebook 格式

# 1. 导入必要的库
import os
import sys
import numpy as np
import h5py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import json
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datetime import datetime
import pickle
import pandas as pd

In [ ]:
# 2. 设置路径和参数
# 【请在这里修改为你的实际路径】
new_data_path = "path/to/your/new_data.h5"  # 新数据集路径
best_model_dir = "path/to/saved/model_dir"  # baseline模型文件夹路径
output_dir = "results/predictions"  # 预测结果保存路径
num_classes = 102  # 类别数量
model_type = "deep_mlp"  # 模型类型: 'mlp' 或 'deep_mlp'
feature_selection = "rf"  # 特征选择方法，根据你的实际情况修改
normalization_method = "standard"  # 标准化方法: 'standard', 'minmax', 或 'robust'
batch_size = 64  # 批量大小

# 创建输出目录
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, timestamp)
os.makedirs(output_path, exist_ok=True)

print(f"输出目录: {output_path}")
print(f"模型目录: {best_model_dir}")
print(f"数据路径: {new_data_path}")

In [ ]:
# 3. 定义数据加载函数
def load_h5_data(file_path):
    """加载HDF5格式的数据"""
    data_dict = {}
    
    print(f"开始加载数据文件: {file_path}")
    with h5py.File(file_path, 'r') as f:
        # 打印H5文件的所有组
        print("H5文件组结构:")
        def visit_for_debug(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  数据集: {name}, 形状: {obj.shape}, 类型: {obj.dtype}")
            else:
                print(f"  组: {name}")
        f.visititems(visit_for_debug)
        
        # 加载数据
        def visit_group(name, obj):
            if isinstance(obj, h5py.Dataset):
                parts = name.split('/')
                current_dict = data_dict
                for i, part in enumerate(parts[:-1]):
                    if part not in current_dict:
                        current_dict[part] = {}
                    current_dict = current_dict[part]
                current_dict[parts[-1]] = obj[()]
                print(f"已加载数据集: {name}")
        
        f.visititems(visit_group)
    
    return data_dict

In [ ]:
# 4. 加载数据并查看结构
data_dict = load_h5_data(new_data_path)

# 打印数据字典的结构
def print_dict_structure(d, prefix=""):
    for k, v in d.items():
        if isinstance(v, dict):
            print(f"{prefix}{k}:")
            print_dict_structure(v, prefix + "  ")
        else:
            print(f"{prefix}{k}: 形状={v.shape if hasattr(v, 'shape') else '标量'}")

print("\n数据字典结构:")
print_dict_structure(data_dict)

In [ ]:
# 5. 提取特征和标签
def extract_features_labels(data_dict, feature_selection=None):
    """从数据字典中提取特征和标签"""
    test_features = None
    test_labels = None
    
    # 尝试多种可能的数据结构路径
    
    # 1. 首先尝试从顶级结构中提取
    if 'test' in data_dict and 'features' in data_dict['test']:
        test_features = data_dict['test']['features']
        test_labels = data_dict['test']['labels'] if 'labels' in data_dict['test'] else None
    
    # 2. 如果上面失败，尝试从 'all' 组提取
    elif 'all' in data_dict:
        if 'test' in data_dict['all']:
            if isinstance(data_dict['all']['test'], dict) and 'features' in data_dict['all']['test']:
                test_features = data_dict['all']['test']['features']
                test_labels = data_dict['all']['test']['labels'] if 'labels' in data_dict['all']['test'] else None
            else:
                test_features = data_dict['all']['test']
                test_labels = data_dict['all']['test_labels'] if 'test_labels' in data_dict['all'] else None
    
    # 3. 如果仍然失败，尝试特征选择路径
    elif feature_selection:
        # 尝试找到包含选定特征选择方法的键
        for key in data_dict.keys():
            if feature_selection in key.lower():
                if 'test' in data_dict[key]:
                    if isinstance(data_dict[key]['test'], dict) and 'features' in data_dict[key]['test']:
                        test_features = data_dict[key]['test']['features']
                        test_labels = data_dict[key]['test']['labels'] if 'labels' in data_dict[key]['test'] else None
                    else:
                        test_features = data_dict[key]['test']
                        test_labels = data_dict[key]['test_labels'] if 'test_labels' in data_dict[key] else None
                    break
    
    # 4. 最后，尝试任何可能的特征路径
    if test_features is None:
        print("无法通过标准路径找到数据，尝试搜索任何可能的路径...")
        
        for key1 in data_dict.keys():
            if test_features is not None:
                break
                
            if isinstance(data_dict[key1], dict):
                for key2 in data_dict[key1].keys():
                    if key2 == 'test' or key2 == 'features' or 'test' in key2 or 'features' in key2:
                        if isinstance(data_dict[key1][key2], np.ndarray):
                            # 可能是特征
                            if len(data_dict[key1][key2].shape) == 2:
                                test_features = data_dict[key1][key2]
                                print(f"找到可能的测试特征: {key1}/{key2} 形状={test_features.shape}")
                                
                                # 尝试找到相关的标签
                                label_keys = [k for k in data_dict[key1].keys() if 'label' in k.lower()]
                                if label_keys:
                                    test_labels = data_dict[key1][label_keys[0]]
                                    print(f"找到可能的测试标签: {key1}/{label_keys[0]} 形状={test_labels.shape}")
                                break
    
    return test_features, test_labels

# 提取特征和标签
test_features, test_labels = extract_features_labels(data_dict, feature_selection)

if test_features is not None:
    print(f"\n成功提取测试特征: {test_features.shape}")
    if test_labels is not None:
        print(f"成功提取测试标签: {test_labels.shape}")
        print(f"标签范围: {np.min(test_labels)} - {np.max(test_labels)}")
        print(f"唯一标签值: {np.unique(test_labels).shape[0]}")
    else:
        print("未找到测试标签")
else:
    print("无法提取测试特征，请检查数据结构")

In [ ]:
# 6. 查找和加载最佳模型文件
def find_best_model(model_dir):
    """在模型目录中查找最佳模型文件"""
    # 先尝试查找名称包含"best_model"的文件
    best_model_files = [f for f in os.listdir(model_dir) if "best_model" in f and f.endswith(".pth")]
    
    if best_model_files:
        # 如果有多个，选择最新的一个
        best_model_files.sort(key=lambda x: os.path.getmtime(os.path.join(model_dir, x)), reverse=True)
        return os.path.join(model_dir, best_model_files[0])
    
    # 如果没有找到，查找任何.pth文件
    pth_files = [f for f in os.listdir(model_dir) if f.endswith(".pth")]
    if pth_files:
        # 选择最新的一个
        pth_files.sort(key=lambda x: os.path.getmtime(os.path.join(model_dir, x)), reverse=True)
        return os.path.join(model_dir, pth_files[0])
    
    return None

# 查找最佳模型文件
best_model_path = find_best_model(best_model_dir)

if best_model_path:
    print(f"找到最佳模型文件: {best_model_path}")
else:
    print(f"在目录 {best_model_dir} 中未找到.pth模型文件")

In [ ]:
# 7. 导入模型类定义
# 需要导入你的模型定义，这里假设模型在项目的models/baseline.py中

# 首先添加项目根目录到系统路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

try:
    # 尝试导入模型定义
    from models.baseline import BaselineMLP, DeepMLP
    print("成功导入模型类")
except ImportError as e:
    print(f"导入模型类失败: {e}")
    print("请确保项目结构正确，或手动添加模型定义")
    
    # 如果导入失败，这里提供模型的简化定义
    print("使用简化的模型定义...")
    
    class BaselineMLP(nn.Module):
        def __init__(self, input_dim, hidden_dims, num_classes, dropout_rate=0.3):
            super(BaselineMLP, self).__init__()
            
            # 构建MLP层
            layers = []
            prev_dim = input_dim
            
            for i, dim in enumerate(hidden_dims):
                layers.append(nn.Linear(prev_dim, dim))
                layers.append(nn.BatchNorm1d(dim))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout_rate))
                prev_dim = dim
            
            self.feature_extractor = nn.Sequential(*layers)
            self.classifier = nn.Linear(prev_dim, num_classes)
        
        def forward(self, x):
            features = self.feature_extractor(x)
            logits = self.classifier(features)
            return logits

    class DeepMLP(nn.Module):
        def __init__(self, input_dim, hidden_dims, num_classes, 
                    use_residual=True, use_self_attention=True, use_feature_interaction=True,
                    dropout_rates=None, num_attn_heads=8, attn_layers=None):
            super(DeepMLP, self).__init__()
            
            self.use_residual = use_residual
            self.use_self_attention = use_self_attention
            self.use_feature_interaction = use_feature_interaction
            
            # 设置dropout率
            if dropout_rates is None:
                dropout_rates = [0.3] * len(hidden_dims)
            
            # 构建网络层
            self.layers = nn.ModuleList()
            prev_dim = input_dim
            
            for i, dim in enumerate(hidden_dims):
                # 添加线性层
                self.layers.append(nn.Linear(prev_dim, dim))
                
                # 添加BN和激活函数
                self.layers.append(nn.Sequential(
                    nn.BatchNorm1d(dim),
                    nn.ReLU(),
                    nn.Dropout(dropout_rates[i])
                ))
                
                prev_dim = dim
            
            # 分类器
            self.classifier = nn.Linear(prev_dim, num_classes)
        
        def forward(self, x):
            for layer in self.layers:
                x = layer(x)
            return self.classifier(x)

In [ ]:
# 8. 加载模型
def load_model(model_path, model_type, input_dim, num_classes):
    """加载预训练模型"""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"使用设备: {device}")
    
    # 创建模型实例
    if model_type == 'mlp':
        model = BaselineMLP(
            input_dim=input_dim,
            hidden_dims=[2048, 1024, 512, 256, 128],
            num_classes=num_classes,
            dropout_rate=0.3
        )
        print("创建BaselineMLP模型")
    elif model_type == 'deep_mlp':
        model = DeepMLP(
            input_dim=input_dim,
            hidden_dims=[2048, 1024, 512, 256, 128],  # 根据你实际的模型配置调整
            num_classes=num_classes,
            use_residual=True,
            use_self_attention=True,
            use_feature_interaction=True
        )
        print("创建DeepMLP模型")
    else:
        raise ValueError(f"不支持的模型类型: {model_type}")
    
    print(f"模型输入维度: {input_dim}")
    print(f"模型输出类别数: {num_classes}")
    
    # 加载预训练参数
    print(f"从 {model_path} 加载模型参数")
    checkpoint = torch.load(model_path, map_location=device)
    
    # 检查checkpoint的结构
    if isinstance(checkpoint, dict):
        print(f"Checkpoint类型: 字典，包含键: {list(checkpoint.keys())}")
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print("从'model_state_dict'加载参数")
        elif 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
            print("从'state_dict'加载参数")
        else:
            model.load_state_dict(checkpoint)
            print("直接从checkpoint字典加载参数")
    else:
        model.load_state_dict(checkpoint)
        print("直接从checkpoint加载参数")
    
    model = model.to(device)
    model.eval()  # 设置为评估模式
    
    return model, device

# 检查是否可以加载模型
if best_model_path and test_features is not None:
    input_dim = test_features.shape[1]  # 使用新数据集的特征维度
    model, device = load_model(best_model_path, model_type, input_dim, num_classes)
    print("模型加载成功")

In [ ]:
# 9. 应用标准化
def apply_standardization(features, scaler_type='standard'):
    """应用标准化处理"""
    print(f"应用{scaler_type}标准化")
    
    # 创建标准化器
    if scaler_type == 'minmax':
        scaler = MinMaxScaler()
    elif scaler_type == 'robust':
        scaler = RobustScaler()
    else:  # 默认使用StandardScaler
        scaler = StandardScaler()
    
    # 拟合并转换数据
    standardized_features = scaler.fit_transform(features)
    
    # 打印标准化前后的统计信息
    print(f"标准化前 - 均值: {np.mean(features):.4f}, 标准差: {np.std(features):.4f}")
    print(f"标准化后 - 均值: {np.mean(standardized_features):.4f}, 标准差: {np.std(standardized_features):.4f}")
    
    # 保存标准化器
    os.makedirs(os.path.join(output_path, 'scalers'), exist_ok=True)
    with open(os.path.join(output_path, 'scalers', f'{scaler_type}_scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)
    print(f"标准化器已保存至 {os.path.join(output_path, 'scalers', f'{scaler_type}_scaler.pkl')}")
    
    return standardized_features, scaler

# 应用标准化
if test_features is not None:
    standardized_features, scaler = apply_standardization(test_features, normalization_method)
    print(f"标准化后的特征维度: {standardized_features.shape}")

In [ ]:
# 10. 进行预测
def predict(model, features, device, batch_size=64):
    """使用模型进行预测"""
    print("开始进行预测...")
    
    # 转换为torch张量
    features_tensor = torch.FloatTensor(features)
    dataset = TensorDataset(features_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    predictions = []
    probabilities = []
    
    with torch.no_grad():
        for batch in dataloader:
            inputs = batch[0].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            
            _, preds = torch.max(outputs, 1)
            
            predictions.extend(preds.cpu().numpy())
            probabilities.extend(probs.cpu().numpy())
    
    return np.array(predictions), np.array(probabilities)

# 进行预测
if 'model' in locals() and standardized_features is not None:
    predictions, probabilities = predict(model, standardized_features, device, batch_size)
    print(f"预测完成，预测形状: {predictions.shape}")
    print(f"概率矩阵形状: {probabilities.shape}")
    
    # 打印预测统计信息
    unique_preds, counts = np.unique(predictions, return_counts=True)
    print(f"预测类别分布:")
    for i, (label, count) in enumerate(zip(unique_preds, counts)):
        if i < 10 or count > len(predictions) * 0.05:  # 只显示前10个或占比>5%的类别
            print(f"  类别 {label}: {count} 个样本 ({count/len(predictions)*100:.2f}%)")
    if len(unique_preds) > 10:
        print(f"  ... 以及其他 {len(unique_preds)-10} 个类别")

In [ ]:
# 11. 评估预测结果（如果有标签）
def evaluate_predictions(predictions, labels):
    """评估预测结果"""
    if labels is None:
        print("未提供标签，跳过评估")
        return None
    
    # 确保标签是整数类型
    labels = labels.astype(np.int64)
    
    # 检查标签范围是否需要调整
    # 如果标签是从1开始的，但预测是从0开始的，需要调整
    if np.min(labels) == 1 and np.min(predictions) == 0:
        print("标签从1开始，预测从0开始，调整标签...")
        labels = labels - 1
    # 如果预测是从1开始的，但标签是从0开始的，需要调整
    elif np.min(labels) == 0 and np.min(predictions) == 1:
        print("标签从0开始，预测从1开始，调整预测...")
        predictions = predictions - 1
    
    # 计算评估指标
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    
    print("\n预测评估结果:")
    print(f"准确率: {accuracy:.4f}")
    print(f"宏平均F1: {f1_macro:.4f}")
    print(f"加权平均F1: {f1_weighted:.4f}")
    
    # 创建分类报告
    report = classification_report(labels, predictions)
    print("\n分类报告:")
    print(report)
    
    # 保存评估结果
    evaluation = {
        'accuracy': float(accuracy),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
        'classification_report': classification_report(labels, predictions, output_dict=True)
    }
    
    with open(os.path.join(output_path, 'evaluation_results.json'), 'w') as f:
        json.dump(evaluation, f, indent=4)
    print(f"评估结果已保存至 {os.path.join(output_path, 'evaluation_results.json')}")
    
    # 绘制混淆矩阵
    plt.figure(figsize=(12, 10))
    cm = confusion_matrix(labels, predictions)
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('混淆矩阵')
    plt.colorbar()
    
    # 限制显示的类别数量，避免混淆矩阵过大
    max_classes_to_show = 20
    if len(np.unique(labels)) > max_classes_to_show:
        print(f"类别数量过多，混淆矩阵仅显示前{max_classes_to_show}个类别")
    
    # 添加数字标签
    classes_to_plot = min(cm.shape[0], max_classes_to_show)
    thresh = cm[:classes_to_plot, :classes_to_plot].max() / 2
    for i in range(classes_to_plot):
        for j in range(classes_to_plot):
            plt.text(j, i, format(cm[i, j], 'd'),
                    horizontalalignment="center",
                    color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, 'confusion_matrix.png'))
    plt.show()
    
    return evaluation

# 如果有标签，评估预测结果
if 'predictions' in locals() and test_labels is not None:
    evaluation = evaluate_predictions(predictions, test_labels)

In [ ]:
# 12. 保存预测结果
def save_predictions(predictions, probabilities, output_dir, labels=None):
    """保存预测结果"""
    print("保存预测结果...")
    
    results = {'prediction': predictions}
    
    # 如果有标签，也保存标签
    if labels is not None:
        # 确保标签与预测的起始索引一致
        if np.min(labels) == 1 and np.min(predictions) == 0:
            results['true_label'] = labels - 1
        elif np.min(labels) == 0 and np.min(predictions) == 1:
            results['true_label'] = labels
        else:
            results['true_label'] = labels
    
    # 只保存前10个类别的概率(如果类别太多)，以避免文件过大
    max_probs_to_save = min(10, probabilities.shape[1])
    for i in range(max_probs_to_save):
        results[f'prob_class_{i}'] = probabilities[:, i]
    
    # 保存最高概率值和对应的类别
    results['max_probability'] = np.max(probabilities, axis=1)
    
    # 转换为DataFrame并保存
    results_df = pd.DataFrame(results)
    csv_path = os.path.join(output_dir, 'predictions.csv')
    results_df.to_csv(csv_path, index=False)
    
    print(f"预测结果已保存至 {csv_path}")
    
    # 显示预测结果的前几行
    print("\n预测结果示例(前5行):")
    print(results_df.head())
    
    return results_df

# 保存预测结果
if 'predictions' in locals() and 'probabilities' in locals():
    results_df = save_predictions(predictions, probabilities, output_path, test_labels)

In [ ]:
# 13. 结果分析和可视化
def visualize_predictions(predictions, probabilities, labels=None):
    """分析和可视化预测结果"""
    print("生成预测结果可视化...")
    
    # 1. 预测类别分布
    plt.figure(figsize=(12, 6))
    unique_preds, counts = np.unique(predictions, return_counts=True)
    
    # 限制显示的类别数量
    max_classes = 20
    if len(unique_preds) > max_classes:
        # 选择样本数最多的前20个类别
        top_indices = np.argsort(counts)[-max_classes:]
        unique_preds = unique_preds[top_indices]
        counts = counts[top_indices]
    
    plt.bar(unique_preds, counts)
    plt.title('预测类别分布')
    plt.xlabel('类别')
    plt.ylabel('样本数')
    plt.xticks(unique_preds)
    plt.savefig(os.path.join(output_path, 'prediction_distribution.png'))
    plt.show()
    
    # 2. 预测概率分布
    plt.figure(figsize=(12, 6))
    max_probs = np.max(probabilities, axis=1)
    plt.hist(max_probs, bins=20)
    plt.title('最高预测概率分布')
    plt.xlabel('概率')
    plt.ylabel('样本数')
    plt.savefig(os.path.join(output_path, 'max_probability_distribution.png'))
    plt.show()
    
    # 3. 如果有标签，分析预测正确和错误的样本
    if labels is not None:
        # 调整标签以匹配预测（如果需要）
        if np.min(labels) == 1 and np.min(predictions) == 0:
            adjusted_labels = labels - 1
        else:
            adjusted_labels = labels
            
        # 计算正确和错误的样本
        correct = (predictions == adjusted_labels)
        accuracy = np.mean(correct)
        
        # 按类别分析准确率
        plt.figure(figsize=(14, 7))
        unique_labels = np.unique(adjusted_labels)
        
        # 限制显示的类别数量
        if len(unique_labels) > max_classes:
            # 选择样本数最多的前20个类别
            label_counts = np.array([np.sum(adjusted_labels == l) for l in unique_labels])
            top_indices = np.argsort(label_counts)[-max_classes:]
            unique_labels = unique_labels[top_indices]
        
        accuracies = []
        sample_counts = []
        
        for label in unique_labels:
            mask = (adjusted_labels == label)
            if np.sum(mask) > 0:
                label_acc = np.mean(correct[mask])
                accuracies.append(label_acc)
                sample_counts.append(np.sum(mask))
            else:
                accuracies.append(0)
                sample_counts.append(0)
        
        # 创建双轴图
        fig, ax1 = plt.subplots(figsize=(14, 7))
        
        # 绘制准确率条形图
        bars = ax1.bar(unique_labels, accuracies, alpha=0.7, label='准确率')
        ax1.set_ylim(0, 1.1)
        ax1.set_ylabel('准确率')
        ax1.set_title('各类别预测准确率和样本数量')
        
        # 添加准确率标签
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}',
                    ha='center', va='bottom')
        
        # 创建第二个Y轴显示样本数量
        ax2 = ax1.twinx()
        ax2.plot(unique_labels, sample_counts, 'ro-', label='样本数量')
        ax2.set_ylabel('样本数量')
        
        # 添加图例
        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines + lines2, labels + labels2, loc='upper right')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, 'class_accuracy.png'))
        plt.show()

# 可视化预测结果
if 'predictions' in locals() and 'probabilities' in locals():
    visualize_predictions(predictions, probabilities, test_labels)

In [ ]:
# 14. 总结
print("\n========== 预测流程总结 ==========")
print(f"数据路径: {new_data_path}")
print(f"模型路径: {best_model_path}")
print(f"输出目录: {output_path}")

if 'test_features' in locals() and test_features is not None:
    print(f"测试特征: {test_features.shape}")
    if 'standardized_features' in locals():
        print(f"标准化后: {standardized_features.shape}")
    
if 'test_labels' in locals() and test_labels is not None:
    print(f"测试标签: {test_labels.shape}")
    print(f"标签范围: {np.min(test_labels)} - {np.max(test_labels)}")

if 'predictions' in locals():
    print(f"预测结果: {predictions.shape}")
    if 'evaluation' in locals() and evaluation:
        print(f"准确率: {evaluation['accuracy']:.4f}")
        print(f"宏平均F1: {evaluation['f1_macro']:.4f}")
        print(f"加权平均F1: {evaluation['f1_weighted']:.4f}")

print("\n生成的文件:")
print(f"- 预测结果CSV: {os.path.join(output_path, 'predictions.csv')}")
if 'test_labels' in locals() and test_labels is not None:
    print(f"- 评估结果JSON: {os.path.join(output_path, 'evaluation_results.json')}")
    print(f"- 混淆矩阵图: {os.path.join(output_path, 'confusion_matrix.png')}")
print(f"- 预测分布图: {os.path.join(output_path, 'prediction_distribution.png')}")
print(f"- 概率分布图: {os.path.join(output_path, 'max_probability_distribution.png')}")
if 'test_labels' in locals() and test_labels is not None:
    print(f"- 类别准确率图: {os.path.join(output_path, 'class_accuracy.png')}")
print(f"- 标准化器: {os.path.join(output_path, 'scalers', f'{normalization_method}_scaler.pkl')}")

print("\n预测流程完成!")